In [ ]:
# --- IMPORTS ---
import os
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# --- DOWNLOAD NLTK DATASETS IF NEEDED ---
nltk.download('punkt')


# --- PREPROCESSING FUNCTION (no stopword removal) ---
def preprocess(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum()]
    return filtered_tokens


# --- LOAD DOCUMENTS ---
def load_documents(directory):
    documents = {}
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                documents[filename] = preprocess(file.read())
    return documents


# --- BUILD BOOLEAN MODEL (1 if term exists, 0 otherwise) ---
def build_boolean_model(documents):
    all_terms = set()
    for doc in documents.values():
        all_terms.update(doc)

    term_document_matrix = {}
    for term in all_terms:
        term_document_matrix[term] = {}
        for doc_name, doc_terms in documents.items():
            term_document_matrix[term][doc_name] = 1 if term in doc_terms else 0
    return term_document_matrix


# --- BOOLEAN RETRIEVAL FUNCTION (AND logic) ---
def boolean_retrieval(term_document_matrix, documents, query_terms):
    if not query_terms:
        return set()

    retrieved_docs = set(documents.keys())
    for term in query_terms:
        if term in term_document_matrix:
            term_docs = {doc for doc, present in term_document_matrix[term].items() if present == 1}
            retrieved_docs &= term_docs  # AND logic
        else:
            return set()  # if any term missing, no documents
    return retrieved_docs


# --- MAIN PIPELINE ---
# Load and preprocess documents
directory = 'news_articles'  # <-- Your folder with articles
documents = load_documents(directory)
doc_names = list(documents.keys())
doc_texts = [' '.join(terms) for terms in documents.values()]  # Join tokens back into strings for CountVectorizer

# Build Boolean Model
boolean_model = build_boolean_model(documents)

# Build Term-Document Matrix (Count Vectorizer)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(doc_texts)

# --- USER INPUT QUERY ---
user_query = input("Enter your search query: ").strip()
query_terms = preprocess(user_query)
query_processed_text = ' '.join(query_terms)
query_vec = vectorizer.transform([query_processed_text])

# --- BOOLEAN RETRIEVAL ---
retrieved_docs_boolean = boolean_retrieval(boolean_model, documents, query_terms)

# --- COSINE RETRIEVAL WITHOUT LSI ---
cosine_similarities_no_lsi = cosine_similarity(query_vec, X).flatten()
ranked_docs_no_lsi = np.argsort(-cosine_similarities_no_lsi)

# --- LSI RETRIEVAL ---
k = 100
svd = TruncatedSVD(n_components=min(k, X.shape[1] - 1))
X_lsi = svd.fit_transform(X)
query_vec_lsi = svd.transform(query_vec)
cosine_similarities_lsi = cosine_similarity(query_vec_lsi, X_lsi).flatten()
ranked_docs_lsi = np.argsort(-cosine_similarities_lsi)

# --- PRINT RESULTS ---
print("=========================")
print("BOOLEAN RETRIEVAL (AND logic):")
#print("=========================")
if retrieved_docs_boolean:
    for doc in retrieved_docs_boolean:
        print(doc)
else:
    print("No documents retrieved with Boolean AND logic.")

print("=========================")
print("TOP 10 DOCUMENTS WITHOUT LSI (Cosine Similarity):")
#print("=========================")
for idx in ranked_docs_no_lsi[:10]:
    print(f"{doc_names[idx]}\t(score: {cosine_similarities_no_lsi[idx]:.4f})")

print("=========================")
print("TOP 10 DOCUMENTS WITH LSI:")
#print("=========================")
for idx in ranked_docs_lsi[:10]:
    print(f"{doc_names[idx]}\t(score: {cosine_similarities_lsi[idx]:.4f})")


In [ ]:
# --- IMPORTS ---
import os
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# --- DOWNLOAD NLTK DATASETS IF NEEDED ---
nltk.download('punkt')
nltk.download('stopwords')


# --- PREPROCESSING FUNCTION ---
def preprocess(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return ' '.join(filtered_tokens)  # Important: join back to string for CountVectorizer


# --- LOAD DOCUMENTS ---
def load_documents(directory):
    documents = {}
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                documents[filename] = preprocess(file.read())
    return documents


# --- MAIN PIPELINE ---
# Load and preprocess documents
directory = 'news_articles'  # <-- make sure your folder name is correct
documents = load_documents(directory)

# Document names and contents
doc_names = list(documents.keys())
doc_texts = list(documents.values())

# Build Term-Document Matrix (Count Vectorizer)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(doc_texts)  # Shape: (n_docs, n_terms)

# --- USER INPUT QUERY ---
user_query = input("Enter your search query: ").strip()
query_processed = preprocess(user_query)
query_vec = vectorizer.transform([query_processed])

# --- 1. BASELINE: VECTOR SPACE RETRIEVAL WITHOUT LSI ---
cosine_similarities_no_lsi = cosine_similarity(query_vec, X).flatten()
ranked_docs_no_lsi = np.argsort(-cosine_similarities_no_lsi)

# --- 2. UPDATED: RETRIEVAL WITH LSI ---
k = 100  # Number of dimensions to keep (adjust if your data is small)
svd = TruncatedSVD(n_components=min(k, X.shape[1] - 1))
X_lsi = svd.fit_transform(X)
query_vec_lsi = svd.transform(query_vec)

cosine_similarities_lsi = cosine_similarity(query_vec_lsi, X_lsi).flatten()
ranked_docs_lsi = np.argsort(-cosine_similarities_lsi)

# --- PRINT COMPARISON ---
print("=========================")
print("Top Documents WITHOUT LSI:")
#print("=========================")
for idx in ranked_docs_no_lsi[:10]:  # Top 10 documents
    print(f"{doc_names[idx]}\t(score: {cosine_similarities_no_lsi[idx]:.4f})")

print("=========================")
print("Top Documents WITH LSI:")
#print("=========================")
for idx in ranked_docs_lsi[:10]:
    print(f"{doc_names[idx]}\t(score: {cosine_similarities_lsi[idx]:.4f})")


In [ ]:
# --- IMPORTS ---
import os
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# --- DOWNLOAD NLTK DATASETS IF NEEDED ---
nltk.download('punkt')


# --- PREPROCESSING FUNCTION (no stopword removal here) ---
def preprocess(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum()]
    return ' '.join(filtered_tokens)


# --- LOAD DOCUMENTS ---
def load_documents(directory):
    documents = {}
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                documents[filename] = preprocess(file.read())
    return documents


# --- MAIN PIPELINE ---
# Load and preprocess documents
directory = 'news_articles'  # <-- Your folder with articles
documents = load_documents(directory)

# Document names and contents
doc_names = list(documents.keys())
doc_texts = list(documents.values())

# Build Term-Document Matrix (Count Vectorizer)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(doc_texts)

# --- USER INPUT QUERY ---
user_query = input("Enter your search query: ").strip()
query_processed = preprocess(user_query)
query_vec = vectorizer.transform([query_processed])

# --- 1. BASELINE: VECTOR SPACE RETRIEVAL WITHOUT LSI ---
cosine_similarities_no_lsi = cosine_similarity(query_vec, X).flatten()

# --- 2. LSI: APPLY SVD ---
k = 100  # number of dimensions to keep
svd = TruncatedSVD(n_components=min(k, X.shape[1] - 1))
X_lsi = svd.fit_transform(X)
query_vec_lsi = svd.transform(query_vec)

cosine_similarities_lsi = cosine_similarity(query_vec_lsi, X_lsi).flatten()

# --- THRESHOLD-BASED RETRIEVAL ---
threshold = 0.1  # minimum cosine similarity to consider retrieval

# Retrieve documents above threshold
retrieved_docs_no_lsi = [(doc_names[idx], cosine_similarities_no_lsi[idx])
                         for idx in np.argsort(-cosine_similarities_no_lsi)
                         if cosine_similarities_no_lsi[idx] > threshold]

retrieved_docs_lsi = [(doc_names[idx], cosine_similarities_lsi[idx])
                      for idx in np.argsort(-cosine_similarities_lsi)
                      if cosine_similarities_lsi[idx] > threshold]

# --- PRINT RESULTS ---

print("=========================")
print(f"Documents WITHOUT LSI (cosine similarity > {threshold}):")
#print("=========================")
if retrieved_docs_no_lsi:
    for doc, score in retrieved_docs_no_lsi:
        print(f"{doc}\t(score: {score:.4f})")
else:
    print("No documents retrieved without LSI.")

print("=========================")
print(f"Documents WITH LSI (cosine similarity > {threshold}):")
#print("=========================")
if retrieved_docs_lsi:
    for doc, score in retrieved_docs_lsi:
        print(f"{doc}\t(score: {score:.4f})")
else:
    print("No documents retrieved with LSI.")
